In [82]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [83]:
path = "online_retail_II.xlsx"

df_2009 = pd.read_excel(path, sheet_name='Year 2009-2010')
df_2010 = pd.read_excel(path, sheet_name='Year 2010-2011')
df_raw = pd.concat([df_2009, df_2010], ignore_index=True)
df = df_raw.copy()
df

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
...,...,...,...,...,...,...,...,...
1067366,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
1067367,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
1067368,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France
1067369,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680.0,France


In [84]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[ns]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 65.1+ MB


In [85]:
print(f"Check missing values: \n{df.isna().sum()}")

Check missing values: 
Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64


# Customer ID

Filling the 243,007 (22.8%) missing IDs with the mode (the most frequent customer) would assign all of those rows to one real customer, giving them a large fake purchase history. The mean is meaningless for an identifier. Churn is defined per customer, so any customer assigned by imputation would carry a fabricated label. 

In [86]:
mixed = df.groupby('Invoice')['Customer ID'].agg(lambda x: x.isna().any() and x.notna().any())
print(mixed.sum())

0


No invoice mixes rows with and without a Customer ID, so the missing IDs cannot be recovered from the same invoice. They are likely guest checkouts, so rows with no Customer ID were removed. The model covers identified customers only.

In [87]:
print(f"Check duplicated rows: \n{df.duplicated().sum()}")

Check duplicated rows: 
34335


# Invoice & Quantity

In [88]:
import re
# Replace every digit with "9" and every letter with "A" so we can see the *shape* of a value (e.g. "85123A" -> "99999A") without caring
# about the digits/letters. Values with the same shape get grouped together, so we can spot which shapes are common and which are suspicious.
def number(s):
      s = re.sub(r'[0-9]', '9', str(s))
      return s

def letter(s):
      s = re.sub(r'[A-Za-z]', 'A', s)
      return s 

# InvoiceNo: Invoice number. Nominal, a 6-digit integral number uniquely assigned to each transaction. 
# If this code starts with letter 'c', it indicates a cancellation. 
print(df['Invoice'].apply(number).value_counts())

Invoice
999999     1047871
C999999      19494
A999999          6
Name: count, dtype: int64


In [89]:
non_c = df[df['Invoice'].astype(str).str.startswith('A')]
print(non_c)

        Invoice StockCode      Description  Quantity         InvoiceDate  \
179403  A506401         B  Adjust bad debt         1 2010-04-29 13:36:00   
276274  A516228         B  Adjust bad debt         1 2010-07-19 11:24:00   
403472  A528059         B  Adjust bad debt         1 2010-10-20 12:04:00   
825443  A563185         B  Adjust bad debt         1 2011-08-12 14:50:00   
825444  A563186         B  Adjust bad debt         1 2011-08-12 14:51:00   
825445  A563187         B  Adjust bad debt         1 2011-08-12 14:52:00   

           Price  Customer ID         Country  
179403 -53594.36          NaN  United Kingdom  
276274 -44031.79          NaN  United Kingdom  
403472 -38925.87          NaN  United Kingdom  
825443  11062.06          NaN  United Kingdom  
825444 -11062.06          NaN  United Kingdom  
825445 -11062.06          NaN  United Kingdom  


**Invoice format check**

Invoice numbers should be 6 digits, with a C prefix for cancellations (per the dataset documentation).
The shape check found 3 formats: 999999 (normal), C999999 (19,494 cancellations) and A999999 (6 rows).

The 6 A-prefix invoices are not documented. All of them have StockCode = B, the Description "Adjust bad debt", Quantity = 1, a large price and no Customer ID. This suggests they are internal accounting adjustments, not customer purchases.
They are not cancellations either, since their Quantity is positive.

Decision: they are removed later with all rows without a Customer ID.

In [90]:
is_c = df['Invoice'].astype(str).str.startswith('C')
is_neg = df['Quantity'] < 0

print("Number of invoice starting with C and quantity is negative", df[is_c & is_neg].shape)
print("Invoice starts with C and its quantity is not negative", df[is_c &  ~is_neg])                          

Number of invoice starting with C and quantity is negative (19493, 8)
Invoice starts with C and its quantity is not negative        Invoice StockCode Description  Quantity         InvoiceDate   Price  \
76799  C496350         M      Manual         1 2010-02-01 08:24:00  373.57   

       Customer ID         Country  
76799          NaN  United Kingdom  


This inconsistent invoice also does not have customer ID, which will be removed.

In [91]:
print("Invoice does not start with C but has negative quantity \n", df[~is_c & is_neg])
print("Invoice does not start with C but has negative quantity and customer ID\n", df[~is_c & is_neg]['Customer ID'].notna().sum())

Invoice does not start with C but has negative quantity 
         Invoice StockCode   Description  Quantity         InvoiceDate  Price  \
263      489464     21733  85123a mixed       -96 2009-12-01 10:52:00    0.0   
283      489463     71477         short      -240 2009-12-01 10:52:00    0.0   
284      489467    85123A   21733 mixed      -192 2009-12-01 10:53:00    0.0   
470      489521     21646           NaN       -50 2009-12-01 11:44:00    0.0   
3114     489655     20683           NaN       -44 2009-12-01 17:26:00    0.0   
...         ...       ...           ...       ...                 ...    ...   
1060794  581210     23395         check       -26 2011-12-07 18:36:00    0.0   
1060796  581212     22578          lost     -1050 2011-12-07 18:38:00    0.0   
1060797  581213     22576         check       -30 2011-12-07 18:38:00    0.0   
1062371  581226     23090       missing      -338 2011-12-08 09:56:00    0.0   
1064386  581422     23169       smashed      -235 2011-12-08 1

In [92]:
print((df[~is_c & is_neg]['Price'] == 0).all())

True


3,457 non-cancellation rows have a negative Quantity. All have Price = 0 and no Customer ID, with descriptions such as "lost", "check", "smashed" and "missing", so they are stock write-offs, not customer transactions. They are removed with the rows without a Customer ID.

In [93]:
# drop invoice without customer ID
df = df[df['Customer ID'].notna()].copy()
print(df.shape)    
print("check missing values\n", df.isna().sum())

(824364, 8)
check missing values
 Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
Price          0
Customer ID    0
Country        0
dtype: int64


# Stock Code

In [94]:
df['StockCode'] = df['StockCode'].astype(str)
df['StockCode'] = df['StockCode'].str.strip()   

In [95]:
# StockCode: Product (item) code. Nominal, a 5-digit integral number uniquely assigned to each distinct product.
# Description: Product (item) name. Nominal.
print(df['StockCode'].apply(number).apply(letter).value_counts())


StockCode
99999           729813
99999A           89631
AAAA              2054
A                 1289
99999AA           1182
A9                 259
AAAAAA              61
AAAA AAAAAAA        38
AAAA999             16
AAA                 16
AAAAAA9              3
AA9999               2
Name: count, dtype: int64


In [96]:
df['Description'] = df['Description'].str.strip()
print(df['Description'].value_counts())

Description
WHITE HANGING HEART T-LIGHT HOLDER     5315
REGENCY CAKESTAND 3 TIER               3777
ASSORTED COLOUR BIRD ORNAMENT          2794
JUMBO BAG RED RETROSPOT                2776
LUNCH BAG  BLACK SKULL.                2161
                                       ... 
SET 36 COLOURING PENCILS DOILEY           1
SET 12 COLOURING PENCILS DOILEY           1
ANGEL ON KNEES METAL POLE 100CM H         1
ZINC PLANT POT HOLDER                     1
F.FAIRY,CANDLE IN GLASS,LILY/VALLEY       1
Name: count, Length: 5257, dtype: int64


In [97]:
# 72801C -> ['S/4 ROSE PINK DINNER CANDLES', '4 ROSE PINK DINNER CANDLES', 'check']
# 72801D -> ['S/4 SKY BLUE DINNER CANDLES', '4 SKY BLUE DINNER CANDLES']
# 72801c -> ['S/4 ROSE PINK DINNER CANDLES', '4 ROSE PINK DINNER CANDLES']
# 72801d -> ['S/4 SKY BLUE DINNER CANDLES', '4 SKY BLUE DINNER CANDLES']

df['StockCode'] = df['StockCode'].str.upper()

In [98]:
desc_variation = df.groupby('StockCode')['Description'].nunique()
print(desc_variation.value_counts().sort_index())
multi_desc = desc_variation[desc_variation > 1].index

for code in multi_desc:
    variants = df[df['StockCode'] == code]['Description'].unique()
    print(code, '->', list(variants))

Description
1    4047
2     549
3      43
4       7
Name: count, dtype: int64
15058A -> ['BLUE WHITE SPOTS GARDEN PARASOL', 'BLUE POLKADOT GARDEN PARASOL']
15058B -> ['PINK WHITE SPOTS GARDEN PARASOL', 'PINK POLKADOT GARDEN PARASOL']
16012 -> ['FOOD/DRINK SPUNGE STICKERS', 'FOOD/DRINK SPONGE STICKERS']
16151A -> ['FLOWER DES BLUE HANDBAG/ORANG HANDL', 'FLOWERS HANDBAG blue and orange']
16156L -> ['WRAP, CAROUSEL', 'WRAP CAROUSEL']
16161U -> ['WRAP,SUKI AND FRIENDS', 'WRAP SUKI AND FRIENDS']
17107D -> ["FLOWER FAIRY,5 SUMMER B'DRAW LINERS", 'FLOWER FAIRY 5 DRAWER LINERS', 'FLOWER FAIRY 5 SUMMER DRAW LINERS']
17129F -> ['DEEP BLUE GLASS GEMS IN BAG', 'BLUE GLASS GEMS IN BAG']
18094C -> ['WHITE/BLUE RECT CERAMIC OIL BURNER', 'WHITE AND BLUE CERAMIC OIL BURNER']
20615 -> ['BLUE SPOTTY PASSPORT COVER', 'BLUE POLKADOT PASSPORT COVER']
20622 -> ['VIPPASSPORT COVER', 'VIP PASSPORT COVER']
20652 -> ['BLUE SPOTTY LUGGAGE TAG', 'BLUE POLKADOT LUGGAGE TAG']
20658 -> ['RED SPOTTY LUGGAGE TAG', 'RED

Many StockCodes have more than one Description (typos, wording variants, and junk such as "check"). Description is not used as a model feature (the features come from StockCode, dates, quantity and price), so it is not standardized. 

In [99]:
df['Shape'] = df['StockCode'].apply(number).apply(letter)
codes = (df[df['Shape'].isin(['99999', '99999A', '99999AA'])]
         .groupby('StockCode')['Description'].first()
         .sort_index())
print(codes.to_string())      

StockCode
10002               INFLATABLE POLITICAL GLOBE
10080                 GROOVY CACTUS INFLATABLE
10109                     BENDY COLOUR PENCILS
10120                             DOGGY RUBBER
10123C                    HEARTS WRAPPING TAPE
10123G                 ARMY CAMO WRAPPING TAPE
10124A             SPOTS ON RED BOOKCOVER TAPE
10124G                ARMY CAMO BOOKCOVER TAPE
10125                  MINI FUNKY DESIGN TAPES
10133             COLOURING PENCILS BROWN TUBE
10134             COLOURING PENCILS BROWN TUBE
10135             COLOURING PENCILS BROWN TUBE
10138                ASSORTED COLOUR JUMBO PEN
11001              ASSTD DESIGN RACING CAR PEN
15030                          FAN BLACK FRAME
15034               PAPER POCKET TRAVELING FAN
15036                ASSORTED COLOURS SILK FAN
15039                           SANDALWOOD FAN
15044A                      PINK PAPER PARASOL
15044B                      BLUE PAPER PARASOL
15044C                    PURPLE PAPER PARASOL
150

99999A (127k rows) and 99999AA (1.4k rows) are valid SKUs: a 5-digit product family plus a variant suffix (colour/design). StockCode is kept unchanged, and a `StockPrefix` column is added for family-level features.

In [100]:
df['StockPrefix'] = df['StockCode'].str.extract(r'^(\d{5})')[0]

In [101]:
other = df[~df['Shape'].isin(['99999', '99999A', '99999AA'])]

print(len(other))
print(other.to_string())      

3738
         Invoice     StockCode                          Description  Quantity         InvoiceDate      Price  Customer ID               Country         Shape StockPrefix
89        489439          POST                              POSTAGE         3 2009-12-01 09:28:00     18.000      12682.0                France          AAAA         NaN
126       489444          POST                              POSTAGE         1 2009-12-01 09:55:00    141.000      12636.0                   USA          AAAA         NaN
173       489447          POST                              POSTAGE         1 2009-12-01 10:10:00    130.000      12362.0               Belgium          AAAA         NaN
625       489526          POST                              POSTAGE         6 2009-12-01 11:50:00     18.000      12533.0               Germany          AAAA         NaN
735      C489535             D                             Discount        -1 2009-12-01 12:11:00      9.000      15299.0        United Kingdom  

In [102]:
#remove invalid stockcode, such as POST, M, D
df = df.dropna()
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 820626 entries, 0 to 1067369
Data columns (total 10 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      820626 non-null  object        
 1   StockCode    820626 non-null  object        
 2   Description  820626 non-null  object        
 3   Quantity     820626 non-null  int64         
 4   InvoiceDate  820626 non-null  datetime64[ns]
 5   Price        820626 non-null  float64       
 6   Customer ID  820626 non-null  float64       
 7   Country      820626 non-null  object        
 8   Shape        820626 non-null  object        
 9   StockPrefix  820626 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(6)
memory usage: 68.9+ MB


In [103]:
df['Customer ID'] = df['Customer ID'].astype(int) #customer Id is integer, not float
df = df.drop(columns=['Shape'])     # helper not feature


# Country

In [104]:
# Country
print(df['Country'].value_counts().sort_index().to_string())
print('Number of customers having orders from more than one country:', (df.groupby('Customer ID')['Country'].nunique() > 1).sum())  

Country
Australia                 1904
Austria                    901
Bahrain                     59
Belgium                   2973
Brazil                      94
Canada                     227
Channel Islands           1652
Cyprus                    1171
Czech Republic              28
Denmark                    792
EIRE                     15946
European Community          58
Finland                    996
France                   13689
Germany                  16931
Greece                     658
Iceland                    253
Israel                     324
Italy                     1496
Japan                      578
Korea                       63
Lebanon                     45
Lithuania                  189
Malta                      291
Netherlands               5023
Nigeria                     29
Norway                    1422
Poland                     529
Portugal                  2431
RSA                        122
Saudi Arabia                10
Singapore                  332


In [105]:
multi = df.groupby('Customer ID')['Country'].nunique()
multi = multi[multi > 1].index
print(df[df['Customer ID'].isin(multi)].groupby('Customer ID')['Country'].unique())

Customer ID
12370           [Austria, Cyprus]
12394          [Belgium, Denmark]
12413             [Spain, France]
12417            [Belgium, Spain]
12422    [Australia, Switzerland]
12423          [Denmark, Belgium]
12429          [Denmark, Austria]
12431        [Belgium, Australia]
12449          [Denmark, Belgium]
12455             [Cyprus, Spain]
12457       [Switzerland, Cyprus]
12652           [Germany, France]
Name: Country, dtype: object


# Price

In [106]:
df.describe()

,Quantity,InvoiceDate,Price,Customer ID
count,820626.000000,820626,820626.000000,820626.000000
mean,12.462513,2011-01-01 22:07:06.688772096,2.966967,15331.181611
min,-80995.000000,2009-12-01 07:45:00,0.000000,12346.000000
25%,2.000000,2010-07-06 12:01:00,1.250000,13982.000000
50%,5.000000,2010-12-03 14:18:00,1.950000,15270.000000
75%,12.000000,2011-07-27 15:16:15,3.750000,16799.000000
max,80995.000000,2011-12-09 12:50:00,649.500000,18287.000000
std,189.373801,NaN,4.452568,1694.638540


In [107]:
zero = df[df['Price'] == 0]
print(len(zero))
print(zero['Invoice'].astype(str).str.startswith('C').sum())   # how many are returns
print(zero.head(20))

61
0
       Invoice StockCode                        Description  Quantity  \
4674    489825     22076                   6 RIBBONS EMPIRE        12   
6781    489998     48185                DOOR MAT FAIRY CAKE         2   
18738   490961     22065      CHRISTMAS PUDDING TRINKET POT         1   
18739   490961     22142        CHRISTMAS CRAFT WHITE FAIRY        12   
32916   492079     85042          ANTIQUE LILY FAIRY LIGHTS         8   
40101   492760     21143     ANTIQUE GLASS HEART DECORATION        12   
47126   493761     79320                    FLAMINGO LIGHTS        24   
48342   493899     22355        CHARLOTTE BAG , SUKI DESIGN        10   
57619   494607     21533          RETRO SPOT LARGE MILK JUG        12   
111348  500073     21662         VINTAGE GLASS COFFEE CADDY         1   
149201  503585     22459       CAST IRON HOOK GARDEN TROWEL         8   
149202  503585     22458         CAST IRON HOOK GARDEN FORK         8   
166143  505083     22376  AIRLINE BAG VINTAGE 

In [108]:
ref = df[df['Price'] > 0].groupby('StockCode')['Price'].median().rename('median_price')
z = zero.join(ref, on='StockCode')
print(z[['Invoice','StockCode','Quantity','median_price']].head(20))
print(z['median_price'].isna().sum())     # zero-price codes with no normal price anywhere

share_free = (df[df['Invoice'].isin(zero['Invoice'])]
              .groupby('Invoice')['Price'].agg(lambda s: (s == 0).mean()))
print(share_free.describe())              # 1.0 = whole invoice is free

       Invoice StockCode  Quantity  median_price
4674    489825     22076        12          1.65
6781    489998     48185         2          7.49
18738   490961     22065         1          1.45
18739   490961     22142        12          1.45
32916   492079     85042         8          4.95
40101   492760     21143        12          1.95
47126   493761     79320        24          4.95
48342   493899     22355        10          0.85
57619   494607     21533        12          4.95
111348  500073     21662         1          5.95
149201  503585     22459         8          2.55
149202  503585     22458         8          2.55
166143  505083     22376         1          4.25
232526  511902     21765         1          7.95
240455  512609     20914         2          2.95
248583  513416     22423         5         12.75
276858  516304     22690         6          7.95
296375  518231     22472         9          4.95
327801  521375     22202         3          3.75
358820  524181    46

60 rows (0.008%) have a unit price of 0. All are positive-quantity lines for identified customers (none are returns), and every one of their StockCodes has a normal non-zero price elsewhere, so these are free items (gifts, samples or replacements), not missing prices. Most sit inside otherwise paid invoices. They add 0 to monetary value, so they are kept. A small number of invoices contain only free lines. 

In [109]:
g = df[df['Price'] > 0].groupby('StockCode')['Price']
spread = pd.DataFrame({'n': g.size(), 'n_prices': g.nunique(),
                       'min': g.min(), 'median': g.median(), 'max': g.max()})
spread['max_over_median'] = spread['max'] / spread['median']

spread['min_over_median'] = spread['min'] / spread['median']   # low side
top = spread.sort_values('max_over_median', ascending=False).head(6).index
for c in top:
    s = df[(df['StockCode'] == c) & (df['Price'] > 0)]
    print(c, s['Description'].mode().iloc[0])
    print(s.groupby('Price').agg(n=('Price', 'size'), first=('InvoiceDate', 'min'),
                                 last=('InvoiceDate', 'max'), avg_qty=('Quantity', 'mean'),
                                 n_cust=('Customer ID', 'nunique')))

print(spread['n_prices'].value_counts().head(8))
print(spread['max_over_median'].describe())
print(spread.sort_values('max_over_median', ascending=False).head(15))

17003 BROCADE RING PURSE
         n               first                last      avg_qty  n_cust
Price                                                                  
0.05     1 2010-10-12 16:03:00 2010-10-12 16:03:00  -700.000000       1
0.14     1 2010-06-30 14:55:00 2010-06-30 14:55:00   720.000000       1
0.16    27 2010-02-07 15:49:00 2011-09-28 12:54:00   709.333333       8
0.19     5 2010-09-03 12:01:00 2010-11-24 11:03:00  4752.000000       1
0.21   185 2009-12-01 16:17:00 2011-06-15 09:33:00    55.886486     128
0.25    13 2011-03-16 11:57:00 2011-10-27 12:13:00   628.615385       4
0.29    85 2011-02-14 12:11:00 2011-08-31 13:37:00    54.105882      67
0.32     3 2011-09-19 13:35:00 2011-10-27 18:10:00   504.000000       2
0.39    80 2011-09-07 15:00:00 2011-12-08 11:25:00    35.400000      62
0.63     3 2011-11-14 15:27:00 2011-12-05 17:17:00     2.333333       1
57.60    3 2011-06-21 14:22:00 2011-07-12 11:07:00    -1.000000       1
22502 PICNIC BASKET WICKER SMALL
      

In [110]:

for code in ['17003', '22502']:
    s = df[df['StockCode'] == code]
    print(s.groupby(['Description', 'Price']).agg(n=('Price', 'size'),
                                                avg_qty=('Quantity', 'mean'),
                                                first=('InvoiceDate', 'min'),
                                                last=('InvoiceDate', 'max')))

                            n      avg_qty               first  \
Description        Price                                         
BROCADE RING PURSE 0.05     1  -700.000000 2010-10-12 16:03:00   
                   0.14     1   720.000000 2010-06-30 14:55:00   
                   0.16    27   709.333333 2010-02-07 15:49:00   
                   0.19     5  4752.000000 2010-09-03 12:01:00   
                   0.21   185    55.886486 2009-12-01 16:17:00   
                   0.25    13   628.615385 2011-03-16 11:57:00   
                   0.29    85    54.105882 2011-02-14 12:11:00   
                   0.32     3   504.000000 2011-09-19 13:35:00   
                   0.39    80    35.400000 2011-09-07 15:00:00   
                   0.63     3     2.333333 2011-11-14 15:27:00   
                   57.60    3    -1.000000 2011-06-21 14:22:00   

                                        last  
Description        Price                      
BROCADE RING PURSE 0.05  2010-10-12 16:03:00  


In [111]:
c = df.loc[(df['StockCode'] == '17003') & (df['Price'] == 57.6), 'Customer ID'].unique()
print(c)
print(df[(df['StockCode'] == '17003') & (df['Customer ID'].isin(c))]
      .sort_values('InvoiceDate')[['Invoice', 'Quantity', 'Price', 'InvoiceDate']])

[16422]
        Invoice  Quantity  Price         InvoiceDate
52785    494216       720   0.21 2010-01-12 11:57:00
84390    497225       720   0.16 2010-02-07 15:49:00
94449    498276       720   0.16 2010-02-17 15:41:00
128302   501621       720   0.16 2010-03-18 11:37:00
180251   506454       720   0.16 2010-04-29 19:05:00
211255   509854       720   0.16 2010-05-26 09:24:00
224902   511112       720   0.16 2010-06-07 10:10:00
246641   513272       720   0.16 2010-06-23 12:39:00
278886   516536       720   0.16 2010-07-21 11:36:00
316251   520183       720   0.16 2010-08-24 14:01:00
355948   523976       700   0.21 2010-09-26 14:00:00
370155   525106       720   0.16 2010-10-04 10:50:00
387738  C526671      -700   0.05 2010-10-12 16:03:00
466475   533362       720   0.16 2010-11-17 10:10:00
535206   537214       720   0.16 2010-12-05 15:35:00
512683   537214       720   0.16 2010-12-05 15:35:00
587132   541479       720   0.16 2011-01-18 11:54:00
597222   542207       720   0.16 2011-

Unit price varies legitimately by time (list-price changes, a short sale) and by quantity (wholesale tiers). Some codes share two products (22502). A few isolated extreme prices sit mostly on returns, and they only affect monetary value, so none are removed nor modified.

# Invoice Date

In [112]:
# 1. Each invoice should have exactly one timestamp
n_dates = df.groupby('Invoice')['InvoiceDate'].nunique()
print((n_dates > 1).sum())

# 2. Days with transactions: look for suspicious gaps
days = df['InvoiceDate'].dt.normalize().drop_duplicates().sort_values()
gaps = days.diff().dt.days
print(gaps.value_counts().sort_index())
print(days[gaps > 3])     # dates after a gap longer than 3 days

65
InvoiceDate
1.0     495
2.0     103
3.0       1
5.0       2
12.0      2
Name: count, dtype: int64
45229    2010-01-04
149402   2010-04-06
567942   2011-01-04
693058   2011-04-26
Name: InvoiceDate, dtype: datetime64[ns]


In [113]:
g = df.groupby('Invoice')['InvoiceDate']
multi = g.nunique() > 1
spread = (g.max() - g.min())[multi]
cross_day = (g.min().dt.date != g.max().dt.date)[multi]

print(spread.describe())
print("invoices spanning different calendar dates:", cross_day.sum())

count                           65
mean     0 days 00:01:00.923076923
std      0 days 00:00:07.442084075
min                0 days 00:01:00
25%                0 days 00:01:00
50%                0 days 00:01:00
75%                0 days 00:01:00
max                0 days 00:02:00
Name: InvoiceDate, dtype: object
invoices spanning different calendar dates: 0


These invoices come from same date, just different by minutes, so keep invoice and invoice date

In [114]:
print(df_2009['InvoiceDate'].agg(['min', 'max']))
print(df_2010['InvoiceDate'].agg(['min', 'max']))

min   2009-12-01 07:45:00
max   2010-12-09 20:01:00
Name: InvoiceDate, dtype: datetime64[ns]
min   2010-12-01 08:26:00
max   2011-12-09 12:50:00
Name: InvoiceDate, dtype: datetime64[ns]


In [115]:
window = (df['InvoiceDate'] >= '2010-12-01') & (df['InvoiceDate'] < '2010-12-10')
d = df.duplicated()
print(d.sum(), window.sum(), (d & window).sum(), (d & ~window).sum())

26403 29476 15058 11345


In [116]:
out_dups = df[df.duplicated(keep=False) & ~window].sort_values(['Invoice', 'StockCode'])
print(out_dups['Invoice'].nunique())
print(out_dups.head(50))
print(out_dups.tail(50))

4228
    Invoice StockCode                        Description  Quantity  \
379  489517     21491    SET OF THREE VINTAGE GIFT WRAPS         1   
391  489517     21491    SET OF THREE VINTAGE GIFT WRAPS         1   
365  489517     21821    GLITTER STAR GARLAND WITH BELLS         1   
386  489517     21821    GLITTER STAR GARLAND WITH BELLS         1   
363  489517     21912           VINTAGE SNAKES & LADDERS         1   
371  489517     21912           VINTAGE SNAKES & LADDERS         1   
394  489517     21912           VINTAGE SNAKES & LADDERS         1   
362  489517     21913     VINTAGE SEASIDE JIGSAW PUZZLES         1   
385  489517     21913     VINTAGE SEASIDE JIGSAW PUZZLES         1   
368  489517     22130    PARTY CONE CHRISTMAS DECORATION         6   
383  489517     22130    PARTY CONE CHRISTMAS DECORATION         6   
367  489517     22319  HAIRCLIPS FORTIES FABRIC ASSORTED        12   
384  489517     22319  HAIRCLIPS FORTIES FABRIC ASSORTED        12   
388  489517    

26,403 exact duplicate rows were dropped (15,058 from the sheet overlap, 11,345 within sheets). Within-sheet repeats are most likely system or entry artefacts, since an online cart would normally increase the quantity of one line. Some may be genuine separate lines, so quantity and spend could be slightly understated. Recency, frequency (distinct invoices) and the churn label are unaffected.

In [117]:
# drop duplicated rows
n_before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(n_before, len(df), n_before - len(df))   
print(df.duplicated().sum())                  
print(df['Invoice'].nunique(), df['Customer ID'].nunique())

820626 794223 26403
0
43879 5876


# Save cleaned data

In [118]:
df.to_csv('cleaned_online_retail.csv', index=False)